In [30]:
import pathlib

BASE_DIR = pathlib.Path().resolve().parent
DATASET_DIR = BASE_DIR / "datasets"
EXPORT_DIR = DATASET_DIR / "exports"
EXPORT_DIR.mkdir(exist_ok=True, parents=True)
SPAM_DATASET_PATH = EXPORT_DIR / "spam-dataset.csv"

ZIPS_DIR = DATASET_DIR / 'zips'
ZIPS_DIR.mkdir(exist_ok=True, parents=True)

SPAM_SMS_ZIP_PATH = ZIPS_DIR / "sms-spam-dataset.zip"
SPAM_YOUTUBE_ZIP_PATH = ZIPS_DIR / "youtube-spam-dataset.zip"

In [31]:
SMS_SPAM_ZIP = "https://archive.ics.uci.edu/ml/machine-learning-databases/00228/smsspamcollection.zip"
YOUTUBE_SPAM_ZIP = "https://archive.ics.uci.edu/ml/machine-learning-databases/00380/YouTube-Spam-Collection-v1.zip"

In [32]:
!curl $SMS_SPAM_ZIP -o $SPAM_SMS_ZIP_PATH

!curl $YOUTUBE_SPAM_ZIP -o $SPAM_YOUTUBE_ZIP_PATH  

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  198k    0  198k    0     0  41669      0 --:--:--  0:00:04 --:--:-- 41674
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  159k    0  159k    0     0  35264      0 --:--:--  0:00:04 --:--:-- 39290


In [33]:
SPAM_CLASSIFIER_DIR = DATASET_DIR / "spam-classifier"
SMS_SPAM_DIR = SPAM_CLASSIFIER_DIR / "spam-sms"
YOUTUBE_SPAM_DIR = SPAM_CLASSIFIER_DIR / "youtube-spam"


SMS_SPAM_DIR.mkdir(exist_ok=True, parents=True)
YOUTUBE_SPAM_DIR.mkdir(exist_ok=True, parents=True)

In [34]:
!unzip -o $SPAM_SMS_ZIP_PATH -d $SMS_SPAM_DIR
!unzip -o $SPAM_YOUTUBE_ZIP_PATH -d $YOUTUBE_SPAM_DIR

Archive:  /home/lenny/PROJECTS/spam-detector/datasets/zips/sms-spam-dataset.zip
  inflating: /home/lenny/PROJECTS/spam-detector/datasets/spam-classifier/spam-sms/SMSSpamCollection  
  inflating: /home/lenny/PROJECTS/spam-detector/datasets/spam-classifier/spam-sms/readme  
Archive:  /home/lenny/PROJECTS/spam-detector/datasets/zips/youtube-spam-dataset.zip
  inflating: /home/lenny/PROJECTS/spam-detector/datasets/spam-classifier/youtube-spam/Youtube01-Psy.csv  
  inflating: /home/lenny/PROJECTS/spam-detector/datasets/spam-classifier/youtube-spam/__MACOSX/._Youtube01-Psy.csv  
  inflating: /home/lenny/PROJECTS/spam-detector/datasets/spam-classifier/youtube-spam/Youtube02-KatyPerry.csv  
  inflating: /home/lenny/PROJECTS/spam-detector/datasets/spam-classifier/youtube-spam/__MACOSX/._Youtube02-KatyPerry.csv  
  inflating: /home/lenny/PROJECTS/spam-detector/datasets/spam-classifier/youtube-spam/Youtube03-LMFAO.csv  
  inflating: /home/lenny/PROJECTS/spam-detector/datasets/spam-classifier/yout

In [57]:
sms_spam_input_path = SMS_SPAM_DIR / "SMSSpamCollection" # tsv
# sms_spam_input_path.read_text()

In [58]:
for path in YOUTUBE_SPAM_DIR.glob("*"):
    print(path)

/home/lenny/PROJECTS/spam-detector/datasets/spam-classifier/youtube-spam/Youtube02-KatyPerry.csv
/home/lenny/PROJECTS/spam-detector/datasets/spam-classifier/youtube-spam/Youtube03-LMFAO.csv
/home/lenny/PROJECTS/spam-detector/datasets/spam-classifier/youtube-spam/Youtube01-Psy.csv
/home/lenny/PROJECTS/spam-detector/datasets/spam-classifier/youtube-spam/Youtube04-Eminem.csv
/home/lenny/PROJECTS/spam-detector/datasets/spam-classifier/youtube-spam/Youtube05-Shakira.csv
/home/lenny/PROJECTS/spam-detector/datasets/spam-classifier/youtube-spam/__MACOSX


In [64]:
import pandas as pd
sms_spam_input_path = SMS_SPAM_DIR / "SMSSpamCollection"

sms_df = pd.read_csv(sms_spam_input_path, sep="\t")
sms_df.columns = ["label", "text"]
sms_df['source'] = "sms-spam"
sms_df

,label,text,source
0,ham,Ok lar... Joking wif u oni...,sms-spam
1,spam,Free entry in 2 a wkly comp to win FA Cup fina...,sms-spam
2,ham,U dun say so early hor... U c already then say...,sms-spam
3,ham,"Nah I don't think he goes to usf, he lives aro...",sms-spam
4,spam,FreeMsg Hey there darling it's been 3 week's n...,sms-spam
...,...,...,...
5566,spam,This is the 2nd time we have tried 2 contact u...,sms-spam
5567,ham,Will ü b going to esplanade fr home?,sms-spam
5568,ham,"Pity, * was in mood for that. So...any other s...",sms-spam
5569,ham,The guy did some bitching but I acted like i'd...,sms-spam


In [65]:
my_dfs = []
for path in YOUTUBE_SPAM_DIR.glob("*csv"):
    raw_df = pd.read_csv(path)
    raw_df.rename(columns={"CLASS":"raw_label","CONTENT":"text"},inplace=True)
    raw_df["label"] = raw_df["raw_label"].apply(lambda x: "spam" if str(x) == "1" else "ham")
    raw_df['raw_source'] = str(path.name)
    raw_df['source'] = "youtube-spam"
    df = raw_df.copy()[['label', 'text', 'source']]
    my_dfs.append(df)
yt_dfs = pd.concat(my_dfs)


In [66]:
yt_dfs.tail(10)

,label,text,source
360,spam,**CHECK OUT MY NEW MIXTAPE**** **CHECK OUT MY ...,youtube-spam
361,spam,**CHECK OUT MY NEW MIXTAPE**** **CHECK OUT MY ...,youtube-spam
362,ham,Waka waka she rules,youtube-spam
363,ham,she is sooooo beautiful!,youtube-spam
364,ham,well done shakira,youtube-spam
365,ham,I love this song because we sing it at Camp al...,youtube-spam
366,ham,I love this song for two reasons: 1.it is abou...,youtube-spam
367,ham,wow,youtube-spam
368,ham,Shakira u are so wiredo,youtube-spam
369,ham,Shakira is the best dancer,youtube-spam


In [67]:
yt_dfs.head(5)

,label,text,source
0,spam,i love this so much. AND also I Generate Free ...,youtube-spam
1,spam,http://www.billboard.com/articles/columns/pop-...,youtube-spam
2,spam,Hey guys! Please join me in my fight to help a...,youtube-spam
3,spam,http://psnboss.com/?ref=2tGgp3pV6L this is the...,youtube-spam
4,spam,Hey everyone. Watch this trailer!!!!!!!! http...,youtube-spam


In [68]:
df = pd.concat([sms_df,yt_dfs])
df

,label,text,source
0,ham,Ok lar... Joking wif u oni...,sms-spam
1,spam,Free entry in 2 a wkly comp to win FA Cup fina...,sms-spam
2,ham,U dun say so early hor... U c already then say...,sms-spam
3,ham,"Nah I don't think he goes to usf, he lives aro...",sms-spam
4,spam,FreeMsg Hey there darling it's been 3 week's n...,sms-spam
...,...,...,...
365,ham,I love this song because we sing it at Camp al...,youtube-spam
366,ham,I love this song for two reasons: 1.it is abou...,youtube-spam
367,ham,wow,youtube-spam
368,ham,Shakira u are so wiredo,youtube-spam


In [69]:
df.to_csv(SPAM_DATASET_PATH, index=False)